# Từ các lớp kết nối đầy đủ đến phép chập (Convolutions)
:label:`sec_why-conv`

Cho đến ngày nay,
các mô hình mà chúng ta đã thảo luận cho đến nay
vẫn là những lựa chọn phù hợp
khi chúng ta làm việc với dữ liệu dạng bảng (tabular data).
Bằng dữ liệu dạng bảng, chúng ta muốn nói rằng dữ liệu bao gồm
các hàng tương ứng với các mẫu thử (examples)
và các cột tương ứng với các đặc trưng (features).
Với dữ liệu dạng bảng, chúng ta có thể dự đoán rằng
các mẫu mà chúng ta tìm kiếm có thể liên quan đến
sự tương tác giữa các đặc trưng,
nhưng chúng ta không giả định bất kỳ cấu trúc nào *a priori* (tiên nghiệm)
về cách các đặc trưng tương tác với nhau.

Đôi khi, chúng ta thực sự thiếu kiến thức để có thể hướng dẫn việc xây dựng các kiến trúc phức tạp hơn.
Trong những trường hợp này, một mạng MLP (Perceptron đa tầng)
có thể là điều tốt nhất mà chúng ta có thể làm.
Tuy nhiên, đối với dữ liệu nhận thức đa chiều (high-dimensional perceptual data),
các mạng không có cấu trúc như vậy có thể trở nên rất cồng kềnh.

Ví dụ, hãy quay lại ví dụ thực tế của chúng ta
về việc phân biệt chó và mèo.
Giả sử chúng ta thực hiện một công việc kỹ lưỡng trong việc thu thập dữ liệu,
thu thập một tập dữ liệu được chú thích gồm các bức ảnh có độ phân giải 1 megapixel.
Điều này có nghĩa là mỗi đầu vào của mạng có 1 triệu chiều.
Ngay cả một sự giảm thiểu mạnh mẽ xuống còn 1000 chiều ẩn
cũng sẽ yêu cầu một lớp kết nối đầy đủ (fully connected layer)
được đặc trưng bởi $10^6 \times 10^3 = 10^9$ tham số.
Trừ khi chúng ta có rất nhiều GPU, có tài năng
về tối ưu hóa phân tán,
và một sự kiên nhẫn phi thường,
việc học các tham số của mạng này
có thể hóa ra là không khả thi.

Một người đọc cẩn thận có thể phản đối lập luận này
trên cơ sở rằng độ phân giải 1 megapixel có thể không cần thiết.
Tuy nhiên, mặc dù chúng ta có thể
tránh được với 100.000 pixel,
lớp ẩn có kích thước 1000 của chúng ta đánh giá thấp một cách thô thiển
số lượng các đơn vị ẩn cần thiết
để học các biểu diễn tốt của hình ảnh,
vì vậy một hệ thống thực tế vẫn sẽ yêu cầu hàng tỷ tham số.
Hơn nữa, việc học một bộ phân loại bằng cách khớp quá nhiều tham số
có thể yêu cầu thu thập một tập dữ liệu khổng lồ.
Và thế mà ngày nay cả con người và máy tính đều có thể
phân biệt chó và mèo khá tốt,
có vẻ như mâu thuẫn với những trực giác này.
Đó là bởi vì hình ảnh thể hiện cấu trúc phong phú
có thể được khai thác bởi con người
và các mô hình học máy.
Mạng thần kinh chập (CNN) là một cách sáng tạo
mà học máy đã áp dụng để khai thác
một số cấu trúc đã biết trong hình ảnh tự nhiên.


## Tính bất biến (Invariance)

Hãy tưởng tượng rằng chúng ta muốn phát hiện một đối tượng trong một hình ảnh.
Có vẻ hợp lý khi bất kỳ phương pháp nào
chúng ta sử dụng để nhận dạng đối tượng không nên quá bận tâm
với vị trí chính xác của đối tượng trong hình ảnh.
Lý tưởng nhất, hệ thống của chúng ta nên khai thác kiến thức này.
Heo thường không bay và máy bay thường không bơi.
Tuy nhiên, chúng ta vẫn nên nhận ra
một con heo nếu nó xuất hiện ở đầu hình ảnh.
Chúng ta có thể lấy một số cảm hứng ở đây
từ trò chơi trẻ em "Where's Waldo"
(bản thân nó đã truyền cảm hứng cho nhiều trò chơi mô phỏng ngoài đời thực, chẳng hạn như trò chơi được mô tả trong :numref:`img_waldo`).
Trò chơi bao gồm một số cảnh hỗn loạn
bùng nổ với các hoạt động.
Waldo xuất hiện ở đâu đó trong mỗi cảnh,
thường ẩn nấp ở một vị trí không ngờ tới.
Mục tiêu của người đọc là tìm ra vị trí của anh ta.
Mặc dù trang phục đặc trưng của anh ta,
điều này có thể khó khăn một cách đáng ngạc nhiên,
do số lượng lớn các yếu tố gây xao nhãng.
Tuy nhiên, *Waldo trông như thế nào*
không phụ thuộc vào *Waldo đang ở đâu*.
Chúng ta có thể quét hình ảnh bằng một bộ phát hiện Waldo
có thể gán một điểm số cho mỗi vùng (patch),
cho biết khả năng vùng đó chứa Waldo.
Trong thực tế, nhiều thuật toán phát hiện và phân vùng đối tượng
dựa trên phương pháp này :cite:`Long.Shelhamer.Darrell.2015`.
Mạng CNN hệ thống hóa ý tưởng này về *tính bất biến không gian* (spatial invariance),
khai thác nó để học các biểu diễn hữu ích
với ít tham số hơn.

![Bạn có thể tìm thấy Waldo (hình ảnh do William Murphy (Infomatique) cung cấp) không?](https://github.com/d2l-ai/d2l-pytorch-colab/blob/master/img/waldo-football.jpg?raw=1)
:width:`400px`
:label:`img_waldo`

Bây giờ chúng ta có thể làm cho những trực giác này cụ thể hơn
bằng cách liệt kê một vài yêu cầu để hướng dẫn thiết kế
của một kiến trúc mạng thần kinh phù hợp cho thị giác máy tính:

1. Trong các lớp đầu tiên, mạng của chúng ta
   nên phản hồi tương tự như nhau đối với cùng một vùng ảnh (patch),
   bất kể nó xuất hiện ở đâu trong hình ảnh. Nguyên lý này được gọi là *tính bất biến tịnh tiến* (translation invariance - hoặc translation equivariance).
1. Các lớp đầu tiên của mạng nên tập trung vào các vùng cục bộ (local regions),
   mà không cần quan tâm đến nội dung của hình ảnh ở các vùng xa xôi. Đây là nguyên lý *tính cục bộ* (locality).
   Cuối cùng, các biểu diễn cục bộ này có thể được tổng hợp
   để đưa ra dự đoán ở cấp độ toàn bộ hình ảnh.
1. Khi chúng ta tiến sâu hơn, các lớp sâu hơn sẽ có thể nắm bắt các đặc trưng phạm vi dài hơn của
   hình ảnh, theo cách tương tự như thị giác cấp cao trong tự nhiên.

Hãy xem điều này chuyển dịch sang toán học như thế nào.


## Ràng buộc mạng MLP

Để bắt đầu, chúng ta có thể xem xét một mạng MLP
với hình ảnh hai chiều $\mathbf{X}$ làm đầu vào
và các biểu diễn ẩn trực tiếp của chúng
$\mathbf{H}$ tương tự được biểu diễn dưới dạng các ma trận (chúng là các tensor hai chiều trong code), trong đó cả $\mathbf{X}$ và $\mathbf{H}$ có cùng hình dạng.
Hãy suy ngẫm về điều đó.
Bây giờ chúng ta tưởng tượng rằng không chỉ các đầu vào mà
cả các biểu diễn ẩn cũng sở hữu cấu trúc không gian.

Gọi $[\mathbf{X}]_{i, j}$ và $[\mathbf{H}]_{i, j}$ biểu thị pixel
tại vị trí $(i,j)$
trong hình ảnh đầu vào và biểu diễn ẩn tương ứng.
Do đó, để có mỗi đơn vị ẩn
nhận đầu vào từ mỗi pixel đầu vào,
chúng ta sẽ chuyển từ việc sử dụng các ma trận trọng số
(như chúng ta đã làm trước đây trong các mạng MLP)
sang việc biểu diễn các tham số của chúng ta
dưới dạng các tensor trọng số bậc bốn $\mathsf{W}$.
Giả sử rằng $\mathbf{U}$ chứa các độ lệch (biases),
chúng ta có thể biểu diễn lớp kết nối đầy đủ một cách chính thức dưới dạng:

$$\begin{aligned} \left[\mathbf{H}\right]_{i, j} &= [\mathbf{U}]_{i, j} + \sum_k \sum_l[\mathsf{W}]_{i, j, k, l}  [\mathbf{X}]_{k, l}\\ &=  [\mathbf{U}]_{i, j} +
\sum_a \sum_b [\mathsf{V}]_{i, j, a, b}  [\mathbf{X}]_{i+a, j+b}.\end{aligned}$$

Sự chuyển đổi từ $\mathsf{W}$ sang $\mathsf{V}$ hoàn toàn mang tính hình thức lúc này
vì có sự tương ứng một-một
giữa các hệ số trong cả hai tensor bậc bốn.
Chúng ta chỉ đơn giản là lập chỉ mục lại các chỉ số dưới $(k, l)$
sao cho $k = i+a$ và $l = j+b$.
Nói cách ứng khác, chúng ta thiết lập $[\mathsf{V}]_{i, j, a, b} = [\mathsf{W}]_{i, j, i+a, j+b}$.
Các chỉ số $a$ và $b$ chạy qua cả độ lệch dương và âm,
bao phủ toàn bộ hình ảnh.
Đối với bất kỳ vị trí ($i$, $j$) nào trong biểu diễn ẩn $[\mathbf{H}]_{i, j}$,
chúng ta tính toán giá trị của nó bằng cách tổng hợp các pixel trong $x$,
tập trung xung quanh $(i, j)$ và được nhân trọng số bởi $[\mathsf{V}]_{i, j, a, b}$. Trước khi tiếp tục, hãy xem xét tổng số tham số cần thiết cho một lớp *đơn lẻ* trong cách tham số hóa này: một hình ảnh $1000 \times 1000$ (1 megapixel) được ánh xạ tới một biểu diễn ẩn $1000 \times 1000$. Điều này đòi hỏi $10^{12}$ tham số, vượt xa những gì máy tính hiện nay có thể xử lý.

### Tính bất biến tịnh tiến (Translation Invariance)

Bây giờ hãy áp dụng nguyên lý đầu tiên
đã được thiết lập ở trên: tính bất biến tịnh tiến :cite:`Zhang.ea.1988`.
Điều này ngụ ý rằng một sự dịch chuyển trong đầu vào $\mathbf{X}$
sẽ dẫn đến một sự dịch chuyển trong biểu diễn ẩn $\mathbf{H}$.
Điều này chỉ có thể xảy ra nếu $\mathsf{V}$ và $\mathbf{U}$ thực sự không phụ thuộc vào $(i, j)$. Như vậy,
chúng ta có $[\mathsf{V}]_{i, j, a, b} = [\mathbf{V}]_{a, b}$ và $\mathbf{U}$ là một hằng số, ví dụ là $u$.
Kết quả là, chúng ta có thể đơn giản hóa định nghĩa cho $\mathbf{H}$:

$$[\mathbf{H}]_{i, j} = u + \sum_a\sum_b [\mathbf{V}]_{a, b}  [\mathbf{X}]_{i+a, j+b}.$$


Đây chính là một phép *chập* (convolution)!
Chúng ta đang nhân trọng số các pixel tại vị trí $(i+a, j+b)$
trong vùng lân cận của vị trí $(i, j)$ với các hệ số $[\mathbf{V}]_{a, b}$
để có được giá trị $[\mathbf{H}]_{i, j}$.
Lưu ý rằng $[\mathbf{V}]_{a, b}$ cần ít hệ số hơn nhiều so với $[\mathsf{V}]_{i, j, a, b}$ vì nó
không còn phụ thuộc vào vị trí trong hình ảnh. Do đó, số lượng tham số yêu cầu không còn là $10^{12}$ mà là một con số hợp lý hơn nhiều: $4 \times 10^6$ (chúng ta vẫn có sự phụ thuộc vào $a, b \in (-1000, 1000)$). Nói tóm lại, chúng ta đã có bước tiến lớn. Mạng thần kinh trễ thời gian (TDNNs) là một trong những ví dụ đầu tiên khai thác ý tưởng này :cite:`Waibel.Hanazawa.Hinton.ea.1989`.

### Tính cục bộ (Locality)

Bây giờ hãy áp dụng nguyên lý thứ hai: tính cục bộ.
Như đã giải thích ở trên, chúng ta tin rằng chúng ta không cần phải
nhìn quá xa vị trí $(i, j)$
để thu thập thông tin liên quan
nhằm đánh giá những gì đang diễn ra tại $[\mathbf{H}]_{i, j}$.
Điều này có nghĩa là bên ngoài một phạm vi $|a|> \Delta$ hoặc $|b| > \Delta$,
chúng ta nên thiết lập $[\mathbf{V}]_{a, b} = 0$.
Tương đương, chúng ta có thể viết lại $[\mathbf{H}]_{i, j}$ thành:

$$[\mathbf{H}]_{i, j} = u + \sum_{a = -\Delta}^{\Delta} \sum_{b = -\Delta}^{\Delta} [\mathbf{V}]_{a, b}  [\mathbf{X}]_{i+a, j+b}.$$
:eqlabel:`eq_conv-layer`

Điều này làm giảm số lượng tham số từ $4 \times 10^6$ xuống còn $4 \Delta^2$, trong đó $\Delta$ thường nhỏ hơn $10$. Như vậy, chúng ta đã giảm số lượng tham số thêm bốn bậc đại lượng (10.000 lần). Lưu ý rằng công thức :eqref:`eq_conv-layer` chính là cái được gọi ngắn gọn là một *lớp chập* (convolutional layer).
*Mạng thần kinh chập* (CNNs)
là một họ mạng thần kinh đặc biệt có chứa các lớp chập.
Trong cộng đồng nghiên cứu deep learning,
$\mathbf{V}$ được gọi là *nhân chập* (convolution kernel),
*bộ lọc* (filter), hoặc đơn giản là *trọng số* (weights) của lớp có thể học được.

Mặc dù trước đây, chúng ta có thể yêu cầu hàng tỷ tham số
để biểu diễn chỉ một lớp duy nhất trong mạng xử lý hình ảnh,
bây giờ chúng ta thường chỉ cần vài trăm tham số, mà không
làm thay đổi số chiều của cả
đầu vào hoặc các biểu diễn ẩn.
Cái giá phải trả cho sự sụt giảm đáng kể các tham số này
là các đặc trưng của chúng ta giờ đây có tính bất biến tịnh tiến
và lớp của chúng ta chỉ có thể kết hợp thông tin cục bộ,
khi xác định giá trị của từng kích hoạt ẩn.
Tất cả việc học đều phụ thuộc vào việc áp dụng thiên kiến quy nạp (inductive bias).
Khi thiên kiến đó phù hợp với thực tế,
chúng ta sẽ có được các mô hình hiệu quả về mặt dữ liệu mẫu
và khái quát hóa tốt cho dữ liệu chưa từng thấy.
Nhưng tất nhiên, nếu các thiên kiến đó không phù hợp với thực tế,
ví dụ, nếu hình ảnh hóa ra không có tính bất biến tịnh tiến,
các mô hình của chúng ta có thể gặp khó khăn ngay cả trong việc khớp với dữ liệu huấn luyện.

Sự cắt giảm đáng kể tham số này đưa chúng ta đến yêu cầu cuối cùng,
đó là các lớp sâu hơn nên biểu diễn các khía cạnh lớn hơn và phức tạp hơn
của một hình ảnh. Điều này có thể đạt được bằng cách xếp chồng xen kẽ các hàm phi tuyến và các lớp chập
nhiều lần.

## Phép chập (Convolutions)

Hãy xem xét ngắn gọn lý do tại sao công thức :eqref:`eq_conv-layer` được gọi là một phép chập.
Trong toán học, *phép chập* giữa hai hàm số :cite:`Rudin.1973`,
ví dụ $f, g: \mathbb{R}^d \to \mathbb{R}$ được định nghĩa là:

$$(f * g)(\mathbf{x}) = \int f(\mathbf{z}) g(\mathbf{x}-\mathbf{z}) d\mathbf{z}.$$

Nghĩa là, chúng ta đo lường mức độ chồng chéo giữa $f$ và $g$
khi một hàm số bị "lật" và dịch chuyển đi một khoảng $\mathbf{x}$.
Bất cứ khi nào chúng ta có các đối tượng rời rạc, tích phân sẽ biến thành một phép tổng.
Ví dụ, đối với các vectơ từ
tập hợp các vectơ vô hạn chiều có tổng bình phương hội tụ
với chỉ số chạy trên $\mathbb{Z}$, chúng ta có được định nghĩa sau:

$$(f * g)(i) = \sum_a f(a) g(i-a).$$

Đối với các tensor hai chiều, chúng ta có một phép tổng tương ứng
với các chỉ số $(a, b)$ cho $f$ và $(i-a, j-b)$ cho $g$:

$$(f * g)(i, j) = \sum_a\sum_b f(a, b) g(i-a, j-b).$$
:eqlabel:`eq_2d-conv-discrete`

Công thức này trông tương tự như :eqref:`eq_conv-layer`, với một sự khác biệt lớn.
Thay vì sử dụng $(i+a, j+b)$, chúng ta lại đang sử dụng phép trừ.
Tuy nhiên, lưu ý rằng sự khác biệt này hầu như chỉ mang tính hình thức
vì chúng ta luôn có thể khớp ký hiệu giữa
:eqref:`eq_conv-layer` và :eqref:`eq_2d-conv-discrete`.
Định nghĩa ban đầu của chúng ta trong :eqref:`eq_conv-layer` mô tả chính xác hơn
một phép *tương quan chéo* (cross-correlation).
Chúng ta sẽ quay lại vấn đề này trong phần tiếp theo.


## Kênh (Channels)
:label:`subsec_why-conv-channels`

Quay trở lại với bộ phát hiện Waldo của chúng ta, hãy xem nó trông như thế nào.
Lớp chập chọn các cửa sổ có kích thước nhất định
và cân đối cường độ theo bộ lọc $\mathsf{V}$, như được thể hiện trong :numref:`fig_waldo_mask`.
Chúng ta có thể hướng tới việc học một mô hình sao cho
bất cứ nơi nào mức độ "giống Waldo" cao nhất,
chúng ta sẽ tìm thấy một đỉnh (peak) trong các biểu diễn lớp ẩn.

![Phát hiện Waldo (hình ảnh do William Murphy (Infomatique) cung cấp).](https://github.com/d2l-ai/d2l-pytorch-colab/blob/master/img/waldo-mask.jpg?raw=1)
:width:`400px`
:label:`fig_waldo_mask`

Chỉ có một vấn đề với cách tiếp cận này.
Cho đến nay, chúng ta đã bỏ qua một thực tế rằng hình ảnh bao gồm
ba kênh màu: đỏ (red), lục (green), và lam (blue).
Tóm lại, hình ảnh không phải là đối tượng hai chiều
mà là các tensor bậc ba,
được đặc trưng bởi chiều cao, chiều rộng và kênh màu,
ví dụ, với hình dạng $1024 \times 1024 \times 3$ pixel.
Trong khi hai trục đầu tiên liên quan đến các mối quan hệ không gian,
trục thứ ba có thể được coi là gán
một biểu diễn đa chiều cho mỗi vị trí pixel.
Do đó, chúng ta lập chỉ mục cho $\mathsf{X}$ dưới dạng $[\mathsf{X}]_{i, j, k}$.
Bộ lọc chập phải điều chỉnh theo đó.
Thay vì $[\mathbf{V}]_{a,b}$, bây giờ chúng ta có $[\mathsf{V}]_{a,b,c}$.

Hơn nữa, giống như đầu vào của chúng ta bao gồm một tensor bậc ba,
nó hóa ra là một ý tưởng tốt nếu chúng ta cũng xây dựng
các biểu diễn ẩn dưới dạng các tensor bậc ba $\mathsf{H}$.
Nói cách khác, thay vì chỉ có một biểu diễn ẩn duy nhất
tương ứng với mỗi vị trí không gian,
chúng ta muốn có một vectơ biểu diễn ẩn hoàn chỉnh
tương ứng với mỗi vị trí không gian.
Chúng ta có thể nghĩ về các biểu diễn ẩn như bao gồm
một số lưới hai chiều xếp chồng lên nhau.
Giống như trong các đầu vào, chúng đôi khi được gọi là các *kênh* (channels).
Chúng cũng đôi khi được gọi là *bản đồ đặc trưng* (feature maps),
vì mỗi bản đồ cung cấp một tập hợp các đặc trưng đã học
được không gian hóa cho lớp tiếp theo.
Trực quan, bạn có thể tưởng tượng rằng ở các lớp thấp hơn gần với đầu vào hơn,
một số kênh có thể chuyên môn hóa để nhận dạng các cạnh trong khi
các kênh khác có thể nhận dạng kết cấu bề mặt.

Để hỗ trợ nhiều kênh trong cả đầu vào ($\mathsf{X}$) và biểu diễn ẩn ($\mathsf{H}$),
chúng ta có thể thêm tọa độ thứ tư vào $\mathsf{V}$: $[\mathsf{V}]_{a, b, c, d}$.
Kết hợp mọi thứ lại với nhau chúng ta có:

$$[\mathsf{H}]_{i,j,d} = \sum_{a = -\Delta}^{\Delta} \sum_{b = -\Delta}^{\Delta} \sum_c [\mathsf{V}]_{a, b, c, d} [\mathsf{X}]_{i+a, j+b, c},$$
:eqlabel:`eq_conv-layer-channels`

trong đó $d$ lập chỉ mục các kênh đầu ra trong biểu diễn ẩn $\mathsf{H}$. Lớp chập tiếp theo sẽ tiếp tục nhận một tensor bậc ba, $\mathsf{H}$, làm đầu vai.
Chúng ta lấy
:eqref:`eq_conv-layer-channels`,
vì tính tổng quát của nó, làm
định nghĩa của một lớp chập cho nhiều kênh, trong đó $\mathsf{V}$ là một nhân hoặc bộ lọc của lớp.

Vẫn còn nhiều hoạt động mà chúng ta cần giải quyết.
Ví dụ, chúng ta cần tìm ra cách kết hợp tất cả các biểu diễn ẩn
thành một đầu ra duy nhất, ví dụ, liệu có Waldo ở *bất kỳ đâu* trong hình ảnh hay không.
Chúng ta cũng cần quyết định cách tính toán mọi thứ hiệu quả,
cách kết hợp nhiều lớp,
các hàm kích hoạt thích hợp,
và cách đưa ra các lựa chọn thiết kế hợp lý
để tạo ra các mạng hiệu quả trong thực tế.
Chúng ta sẽ chuyển sang những vấn đề này trong phần còn lại của chương.

## Tóm tắt và Thảo luận

Trong phần này, chúng ta đã rút ra cấu trúc của mạng thần kinh chập từ các nguyên lý cơ bản. Mặc dù không rõ liệu đây có phải là lộ trình được thực hiện để phát minh ra CNN hay không, thật thỏa mãn khi biết rằng chúng là lựa chọn *đúng đắn* khi áp dụng các nguyên lý hợp lý vào cách các thuật toán xử lý hình ảnh và thị giác máy tính hoạt động, ít nhất là ở các cấp độ thấp hơn. Đặc biệt, tính bất biến tịnh tiến trong hình ảnh ngụ ý rằng tất cả các vùng của một hình ảnh sẽ được xử lý theo cùng một cách. Tính cục bộ có nghĩa là chỉ một vùng lân cận nhỏ của các pixel sẽ được sử dụng để tính toán các biểu diễn ẩn tương ứng. Một số tài liệu tham khảo sớm nhất về CNN là dưới dạng Neocognitron :cite:`Fukushima.1982`.

Nguyên lý thứ hai mà chúng ta gặp phải trong lập luận của mình là làm thế nào để giảm số lượng tham số trong một lớp hàm mà không làm hạn chế khả năng biểu diễn của nó, ít nhất là bất cứ khi nào các giả định nhất định trên mô hình được giữ vững. Chúng ta đã thấy một sự giảm thiểu đáng kể về độ phức tạp do hạn chế này, biến các bài toán không khả thi về mặt tính toán và thống kê thành các mô hình có thể xử lý được.

Việc thêm các kênh cho phép chúng ta mang lại một số độ phức tạp đã bị mất do các hạn chế áp đặt lên nhân chập bởi tính cục bộ và tính bất biến tịnh tiến. Lưu ý rằng việc thêm các kênh khác ngoài đỏ, lục và lam là rất tự nhiên. Nhiều hình ảnh vệ tinh, đặc biệt là trong nông nghiệp và khí tượng học, có hàng chục đến hàng trăm kênh, tạo ra các hình ảnh siêu phổ (hyperspectral images). Chúng báo cáo dữ liệu trên nhiều bước sóng khác nhau. Trong phần tiếp theo, chúng ta sẽ thấy cách sử dụng các phép chập hiệu quả để thao tác trên số chiều của hình ảnh mà chúng hoạt động, cách chuyển từ biểu diễn dựa trên vị trí sang biểu diễn dựa trên kênh, và cách xử lý hiệu quả với số lượng lớn các phân loại.

## Bài tập

1. Giả sử rằng kích thước của nhân chập là $\Delta = 0$.
   Hãy chứng minh rằng trong trường hợp này, nhân chập
   thực hiện một mạng MLP độc lập cho từng tập hợp các kênh. Điều này dẫn đến kiến trúc Mạng trong Mạng (Network in Network) :cite:`Lin.Chen.Yan.2013`.
1. Dữ liệu âm thanh thường được biểu diễn dưới dạng chuỗi một chiều.
    1. Khi nào bạn muốn áp dụng tính cục bộ và tính bất biến tịnh tiến cho âm thanh?
    1. Hãy thiết lập các phép toán chập cho âm thanh.
    1. Bạn có thể xử lý âm thanh bằng các công cụ tương tự như thị giác máy tính không? Gợi ý: sử dụng phổ âm (spectrogram).
1. Tại sao tính bất biến tịnh tiến có thể không phải là một ý tưởng tốt trong một số trường hợp? Cho một ví dụ.
1. Bạn có nghĩ rằng các lớp chập cũng có thể áp dụng cho dữ liệu văn bản không?
   Những vấn đề nào bạn có thể gặp phải với ngôn ngữ?
1. Điều gì xảy ra với các phép chập khi một đối tượng nằm ở biên của một hình ảnh?
1. Chứng minh rằng phép chập có tính chất đối xứng, nghĩa là $f * g = g * f$.

[Thảo luận](https://discuss.d2l.ai/t/64)